# GSV Collection Pipeline

Collects summer Google Street View images for a city in three steps:

**Part 1 — OSM Road Sampling**  
Reads an OpenStreetMap roads shapefile and samples points at 30 m intervals.

**Part 2 — Panorama Metadata Collection**  
Queries the `streetview` package to discover available panoramas near each road point.

**Part 3 — Filter & Download**  
Month filter, highway filter, image download, and GVI screening.

> Paper: *Seasonal GVI Estimation Without Seasonal Imagery: A Zero-Shot MLLM Framework*
> *(SIGSPATIAL '26)* — Code: https://github.com/sjooha111/summer-to-seasonal-GVI  
> GSV images cannot be redistributed (Google ToS).

In [ ]:
%pip install -q streetview geopandas shapely requests tqdm transformers torch torchvision Pillow pandas

## Step 0 — Configuration

**Edit only this cell** before running.  
All three parts share this single CONFIG cell.

In [ ]:
# ── Part 1: OSM Road Sampling ─────────────────────────────────────────────
SHP_PATH   = ""              # Path to OSM roads shapefile (gis_osm_roads_free_1.shp)
CITY       = "Seoul"
COUNTRY    = "South Korea"
HEMISPHERE = "north"         # "north" (summer = Jun–Sep) | "south" (summer = Nov–Feb)
UTM_CRS    = "EPSG:32652"   # UTM EPSG for your city
INTERVAL_M = 30              # Sampling interval along roads (meters)
EXCLUDE_ROAD_TYPES = [       # OSM fclass values to skip
    "motorway", "motorway_link", "footway",
    "path", "steps", "cycleway", "track",
]

# ── Part 2: Metadata ────────────────────────────────────────────────────────
GOOGLE_MAPS_API_KEY = ""     # Required for date supplement (Part 2) and image download (Part 3)

# ── Part 3: Image Download ──────────────────────────────────────────────────
GVI_THRESHOLD  = 0.01        # Skip panoramas whose 4-heading avg GVI falls below this
BATCH_SIZE     = 50          # Stop after this many points for manual review
BUFFER_M       = 20          # Highway exclusion buffer (meters)
HIGHWAY_CLASSES = {"motorway", "motorway_link"}

# ── Output ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = "results"

---
## Part 1 — OSM Road Sampling

Reads the OSM roads shapefile, removes unwanted road types, and samples a point every
`INTERVAL_M` metres along each remaining road segment.  
Output: `road_points.csv` (`point_id`, `lat`, `lon`)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

if not SHP_PATH:
    raise ValueError("Set SHP_PATH to the OSM roads shapefile (.shp)")

roads = gpd.read_file(SHP_PATH)
print(f"Loaded {len(roads):,} road segments")

# Remove excluded road types
if "fclass" in roads.columns:
    roads = roads[~roads["fclass"].isin(EXCLUDE_ROAD_TYPES)]
print(f"After type filter: {len(roads):,} segments")

# Reproject to UTM for metric distances
roads = roads.to_crs(UTM_CRS)

# Sample points at INTERVAL_M intervals along each road
pts = []
for geom in roads.geometry:
    if geom is None:
        continue
    for geom_part in (geom.geoms if hasattr(geom, 'geoms') else [geom]):
        length = geom_part.length
        for d in np.arange(0, length + INTERVAL_M, INTERVAL_M):
            pts.append(geom_part.interpolate(min(d, length)))

gdf = gpd.GeoDataFrame(geometry=pts, crs=UTM_CRS).to_crs("EPSG:4326")
df_pts = pd.DataFrame({
    "point_id": range(len(gdf)),
    "lat":      gdf.geometry.y.round(8),
    "lon":      gdf.geometry.x.round(8),
})

pts_path = OUTPUT_PATH / "road_points.csv"
df_pts.to_csv(pts_path, index=False, encoding="utf-8")
print(f"Road points saved -> {pts_path}  ({len(df_pts):,} points)")
df_pts.head(3)

---
## Part 2 — Panorama Metadata Collection

Calls `streetview.search_panoramas()` for each road point to discover nearby panoramas.
Saves a checkpoint every 500 rows so the cell can be safely re-run after interruption.

> Typical throughput: ~4 points/second → ~100 k points ≈ 7 hours.

In [ ]:
import time
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from streetview import search_panoramas


def collect_streetview_meta(points_df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    """Collect GSV panorama metadata for all road points. Supports resume."""
    out_csv = out_dir / "metadata.csv"
    results, done_panos = [], set()
    if out_csv.exists():
        existing    = pd.read_csv(out_csv)
        done_panos  = set(existing["pano_id"])
        results     = existing.to_dict("records")
        print(f"  Resuming: {len(results):,} existing entries")

    for i, row in enumerate(tqdm(points_df.itertuples(), total=len(points_df),
                                desc="Collecting metadata")):
        try:
            panos = search_panoramas(lat=float(row.lat), lon=float(row.lon))
            for p in (panos or []):
                if p.pano_id in done_panos:
                    continue
                date_str = p.date or ""
                if date_str and len(date_str) == 7:
                    date_str += "-01"
                results.append({
                    "pano_id": p.pano_id,
                    "lat":     round(p.lat, 8),
                    "lon":     round(p.lon, 8),
                    "date":    date_str,
                })
                done_panos.add(p.pano_id)
        except Exception:
            pass
        time.sleep(0.1)
        if (i + 1) % 500 == 0:
            pd.DataFrame(results).drop_duplicates("pano_id").to_csv(out_csv, index=False)

    df = pd.DataFrame(results).drop_duplicates("pano_id")
    df.to_csv(out_csv, index=False)
    print(f"Metadata saved -> {out_csv}  ({len(df):,} panoramas)")
    return df


df_pts  = pd.read_csv(Path(OUTPUT_DIR) / "road_points.csv")
meta_df = collect_streetview_meta(df_pts, Path(OUTPUT_DIR))
meta_df.head(3)

### Step 2b — Supplement Missing Dates (optional)

Fills in missing dates via `get_panorama_meta()`. Skip if `GOOGLE_MAPS_API_KEY` is not set — undated panoramas are dropped in Step 4.

In [ ]:
from streetview import get_panorama_meta
import time, pandas as pd
from pathlib import Path
from tqdm import tqdm

df = pd.read_csv(Path(OUTPUT_DIR) / "metadata.csv")
mask = df["date"].isna() | (df["date"].astype(str).str.strip() == "")
empty_idx = df[mask].index
print(f"Panoramas missing date: {len(empty_idx):,}")

if GOOGLE_MAPS_API_KEY and len(empty_idx) > 0:
    for idx in tqdm(empty_idx, desc="Date supplement"):
        pid = df.at[idx, "pano_id"]
        try:
            meta     = get_panorama_meta(pano_id=pid, api_key=GOOGLE_MAPS_API_KEY)
            date_str = meta.date or ""
            if date_str and len(date_str) == 7:
                date_str += "-01"
            df.at[idx, "date"] = date_str
        except Exception:
            pass
        time.sleep(0.1)
    out = Path(OUTPUT_DIR) / "metadata_dated.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"Saved -> {out}")
else:
    print("Skipped (no API key or no missing dates)")
    df.to_csv(Path(OUTPUT_DIR) / "metadata_dated.csv", index=False, encoding="utf-8")

---
## Part 3 — Filter & Download Summer Images

Applies two pre-download filters then downloads images:
1. **Month filter** — retains summer panoramas only (north: Jun–Sep; south: Nov–Feb)
2. **Highway filter** — excludes panoramas within `BUFFER_M` m of motorways
3. **Download** — fetches 4 headings (0°, 90°, 180°, 270°) via the Static API
4. **GVI check** — runs Mask2Former on downloaded images; skips low-vegetation panoramas

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from pathlib import Path

df = pd.read_csv(Path(OUTPUT_DIR) / "metadata_dated.csv")
print(f"Total panoramas: {len(df):,}")

# ── Month filter ────────────────────────────────────────────────────────────
summer_months = {11, 12, 1, 2} if HEMISPHERE == "south" else {6, 7, 8, 9}
df["month"] = pd.to_datetime(df["date"], errors="coerce").dt.month
df = df[df["month"].isin(summer_months)].copy()
print(f"After month filter ({sorted(summer_months)}): {len(df):,}")

# ── Highway filter ───────────────────────────────────────────────────────────
if SHP_PATH:
    roads     = gpd.read_file(SHP_PATH)
    motorways = roads[roads["fclass"].isin(HIGHWAY_CLASSES)]
    if not motorways.empty:
        buffer = (
            motorways.to_crs(UTM_CRS)
                     .buffer(BUFFER_M)
                     .to_crs("EPSG:4326")
                     .pipe(unary_union)
        )
        gdf = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326"
        )
        before = len(df)
        df     = gdf[~gdf.geometry.intersects(buffer)].drop(columns="geometry").reset_index(drop=True)
        print(f"After highway filter: {len(df):,}  (removed {before - len(df)})")

out = Path(OUTPUT_DIR) / "metadata_filtered.csv"
df.to_csv(out, index=False, encoding="utf-8")
print(f"Filtered metadata saved -> {out}")
df.head(3)

### Step 5 — Load Mask2Former for GVI Screening

GPU recommended; CPU works but is slower.

In [ ]:
import re
import torch
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

MODEL_ID      = "facebook/mask2former-swin-large-ade-semantic"
seg_processor = AutoImageProcessor.from_pretrained(MODEL_ID, size={"height": 640, "width": 640})
seg_model     = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_ID).to(device).eval()

VEG_KEYWORDS = ['tree', 'palm', 'grass', 'plant', 'flower', 'field']
_id2label    = {int(k): v for k, v in seg_model.config.id2label.items()}
VEG_IDS      = [
    i for i, lbl in _id2label.items()
    if any(re.search(r'\b' + kw + r'\b', lbl.lower()) for kw in VEG_KEYWORDS)
]
print(f"Vegetation class IDs: { {_id2label[i]: i for i in VEG_IDS} }")


def compute_vegetation_ratio(image_path) -> float:
    """Return fraction of pixels classified as vegetation (0.0–1.0)."""
    img    = Image.open(image_path).convert("RGB")
    inputs = seg_processor(img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = seg_model(**inputs)
    seg = seg_processor.post_process_semantic_segmentation(
        outputs, target_sizes=[(640, 640)]
    )[0].cpu().numpy()
    veg_pixels = sum((seg == cid).sum() for cid in VEG_IDS)
    return float(veg_pixels) / seg.size

### Step 6 — Download Helpers

In [ ]:
import json, time, requests
from pathlib import Path

BASE_URL      = "https://maps.googleapis.com/maps/api/streetview"
META_URL      = "https://maps.googleapis.com/maps/api/streetview/metadata"
STATIC_SIZE   = (640, 640)
STATIC_FOV    = 90
STATIC_PITCH  = 0
REQUEST_PAUSE = 0.2
HEADINGS      = [0, 90, 180, 270]


def _download_summer(pano_id: str, save_dir: Path) -> bool:
    """Download 4-heading summer images. Returns True if all 4 succeed."""
    save_dir.mkdir(parents=True, exist_ok=True)
    ok = 0
    for heading in HEADINGS:
        params = {
            "pano":    pano_id,
            "size":    f"{STATIC_SIZE[0]}x{STATIC_SIZE[1]}",
            "fov":     STATIC_FOV,
            "pitch":   STATIC_PITCH,
            "heading": heading,
            "key":     GOOGLE_MAPS_API_KEY,
        }
        try:
            resp = requests.get(BASE_URL, params=params, timeout=15)
            if resp.status_code == 200 and len(resp.content) > 5000:
                fname = save_dir / f"summer_{pano_id}_{heading}.jpg"
                fname.write_bytes(resp.content)
                ok += 1
                time.sleep(REQUEST_PAUSE)
            else:
                print(f"  FAIL [{resp.status_code}] heading={heading}")
                break
        except Exception as e:
            print(f"  ERROR heading={heading}: {e}")
            break
    return ok == 4


def _load_progress(prog_file: Path) -> dict:
    if prog_file.exists():
        with open(prog_file, encoding="utf-8") as f:
            return json.load(f)
    return {"processed_count": 0, "results": {}}


def _save_progress(prog: dict, prog_file: Path):
    with open(prog_file, "w", encoding="utf-8") as f:
        json.dump(prog, f, ensure_ascii=False, indent=2)

### Step 7 — Download Loop

Stops automatically after `BATCH_SIZE` points for manual review.  
Re-run the cell to continue — already-processed panoramas are skipped.

In [ ]:
import pandas as pd
from pathlib import Path

if not GOOGLE_MAPS_API_KEY:
    raise ValueError("Set GOOGLE_MAPS_API_KEY in the CONFIG cell")

df_filtered = pd.read_csv(Path(OUTPUT_DIR) / "metadata_filtered.csv")
IMAGES_ROOT = Path("images") / CITY.lower().replace(" ", "_")
PROG_FILE   = Path(OUTPUT_DIR) / f"progress_{CITY.lower()}.json"

all_ids  = df_filtered["pano_id"].unique().tolist()
progress = _load_progress(PROG_FILE)
done_set = set(progress["results"].keys())
base_cnt = progress["processed_count"]
pending  = [pid for pid in all_ids if pid not in done_set]

print(f"Total: {len(all_ids):,} | Done: {len(done_set):,} | Pending: {len(pending):,}")
print(f"This batch: up to {BATCH_SIZE} points\n")

batch = 0
for pano_id in pending:
    if batch >= BATCH_SIZE:
        print(f"\n{'='*55}")
        print(f"[PAUSE] {BATCH_SIZE} points processed — review images then re-run.")
        print(f"Progress file: {PROG_FILE}")
        break

    save_dir = IMAGES_ROOT / pano_id
    print(f"[{base_cnt + batch + 1:4d}] {pano_id}")

    # ── Download 4-heading summer images
    if not _download_summer(pano_id, save_dir):
        print("  FAIL download -> skip")
        progress["results"][pano_id] = {"summer": None, "complete": False}
        batch += 1
        progress["processed_count"] = base_cnt + batch
        _save_progress(progress, PROG_FILE)
        continue

    # ── GVI check via Mask2Former
    veg_ratios = {}
    for heading in HEADINGS:
        img_path = save_dir / f"summer_{pano_id}_{heading}.jpg"
        if img_path.exists():
            veg_ratios[heading] = compute_vegetation_ratio(img_path)

    avg_gvi = sum(veg_ratios.values()) / len(veg_ratios) if veg_ratios else 0.0
    print(f"  avg GVI: {avg_gvi:.4f}")

    if avg_gvi < GVI_THRESHOLD:
        print(f"  SKIP: low vegetation ({avg_gvi:.4f} < {GVI_THRESHOLD}) -> deleting images")
        for f in save_dir.glob("summer_*.jpg"):
            f.unlink()
        progress["results"][pano_id] = {"summer": pano_id, "vegetation_skip": True, "complete": False}
    else:
        print("  OK")
        progress["results"][pano_id] = {"summer": pano_id, "vegetation_skip": False, "complete": True}

    batch += 1
    progress["processed_count"] = base_cnt + batch
    _save_progress(progress, PROG_FILE)

# ── Summary
res      = progress["results"]
total    = len(res)
complete = sum(1 for r in res.values() if r.get("complete"))
veg_skip = sum(1 for r in res.values() if r.get("vegetation_skip"))
print(f"\n{"="*55}")
print(f"Summary [{CITY}] — {total} panoramas processed")
print(f"  Complete (saved): {complete}")
print(f"  Vegetation skip : {veg_skip}")
print(f"  Failed download : {total - complete - veg_skip}")